In [5]:
import re

def parse_expression(expr_str):
    """
    Parses a string like 'Eats(x, Apple)' or 'p(f(a), g(Y))' into a nested list structure.
    Examples:
      'Eats(x, Apple)' -> ['Eats', 'x', 'Apple']
      'p(f(a), g(Y))'  -> ['p', ['f', 'a'], ['g', 'Y']]
    """
    expr_str = expr_str.strip().replace(" ", "")
    if not expr_str:
        return None

    # Match a component: either a nested predicate/function token like name(...) or a simple term
    # This regex pulls out the name and the contents inside the parenthesis safely
    match = re.match(r"^([a-zA-Z0-9_]+)\((.*)\)$", expr_str)
    if not match:
        return expr_str  # It's a plain variable or constant

    name = match.group(1)
    args_str = match.group(2)

    # Split arguments tracking nested parentheses
    args = []
    current_arg = []
    paren_depth = 0

    for char in args_str:
        if char == ',' and paren_depth == 0:
            args.append("".join(current_arg).strip())
            current_arg = []
        else:
            if char == '(':
                paren_depth += 1
            elif char == ')':
                paren_depth -= 1
            current_arg.append(char)

    if current_arg:
        args.append("".join(current_arg).strip())

    return [name] + [parse_expression(arg) for arg in args]

def format_expression(expr_list):
    """Converts a nested list back into a readable string format."""
    if not isinstance(expr_list, list):
        return str(expr_list)
    name = expr_list[0]
    args = ", ".join(format_expression(arg) for arg in expr_list[1:])
    return f"{name}({args})"

def is_variable(x):
    """Variables start with a lowercase letter."""
    return isinstance(x, str) and len(x) > 0 and x[0].islower()

def is_compound(x):
    """Compound terms or predicates are represented as lists."""
    return isinstance(x, list)

def apply_substitution(subst, expression):
    """Recursively applies substitutions to variables in an expression."""
    if is_variable(expression):
        # Keep substituting until no further assignments exist for the resolved variable
        while expression in subst:
            resolved = subst[expression]
            if resolved == expression:
                break
            expression = resolved
        if is_compound(expression):
            return apply_substitution(subst, expression)
        return expression
    elif is_compound(expression):
        return [expression[0]] + [apply_substitution(subst, arg) for arg in expression[1:]]
    return expression

def unify(e1, e2, subst=None):
    if subst is None:
        subst = {}

    if e1 == e2:
        return subst

    if is_variable(e1):
        return unify_variable(e1, e2, subst)

    if is_variable(e2):
        return unify_variable(e2, e1, subst)

    if is_compound(e1) and is_compound(e2):
        # Condition 1 & 2: Predicate/Function names and argument lengths must match
        if e1[0] != e2[0] or len(e1) != len(e2):
            return None

        # Unify arguments one by one
        for arg1, arg2 in zip(e1[1:], e2[1:]):
            subst = unify(arg1, arg2, subst)
            if subst is None:
                return None
        return subst

    return None

def unify_variable(var, val, subst):
    if var in subst:
        return unify(subst[var], val, subst)

    if is_variable(val) and val in subst:
        return unify(var, subst[val], subst)

    # Condition 3: Occur Check (Variable cannot contain itself)
    if occur_check(var, val, subst):
        return None

    new_subst = subst.copy()
    new_subst[var] = val
    return new_subst

def occur_check(var, val, subst):
    if var == val:
        return True
    if is_variable(val) and val in subst:
        return occur_check(var, subst[val], subst)
    if is_compound(val):
        return any(occur_check(var, arg, subst) for arg in val[1:])
    return False


# --- INTERACTIVE USER INPUT INTERFACE ---
if __name__ == "__main__":
    print("=" * 60)
    print("         FIRST-ORDER LOGIC UNIFICATION ENGINE")
    print("=" * 60)
    print("Rules:")
    print("1. Format predicates/functions as: Name(arg1, arg2)")
    print("2. Variables MUST start with a LOWERCASE letter (e.g., x, y, variable1)")
    print("3. Constants/Predicates MUST start with an UPPERCASE letter (e.g., John, Apple, Riya)")
    print("=" * 60)

    # Gather user inputs
    input_A = input("Enter Expression A (e.g., Eats(x, Apple)): ").strip()
    input_B = input("Enter Expression B (e.g., Eats(Riya, y)): ").strip()
    print("-" * 60)

    try:
        parsed_A = parse_expression(input_A)
        parsed_B = parse_expression(input_B)

        if not isinstance(parsed_A, list) or not isinstance(parsed_B, list):
            print("Error: Please provide valid compound expressions like P(x) or Eats(x, y).")
        else:
            # Execute Unification
            mgu = unify(parsed_A, parsed_B)

            if mgu is not None:
                print("Result: UNIFICATION SUCCESS ✅\n")
                print("Most General Unifier (MGU) Substitutions:")
                if not mgu:
                    print("  None (Expressions were already identical)")
                for var, val in mgu.items():
                    print(f"  {var} -> {format_expression(val)}")

                print("\nFinal Unified Expression:")
                unified_expr = apply_substitution(mgu, parsed_A)
                print(f"  {format_expression(unified_expr)}")
            else:
                print("Result: UNIFICATION FAILED ❌")
                print("Reason: Predicates mismatched, argument lengths differed, or variable assignment clashed.")
    except Exception as e:
        print(f"Parsing Error: Please check your syntax/parentheses format. ({e})")
    print("=" * 60)


         FIRST-ORDER LOGIC UNIFICATION ENGINE
Rules:
1. Format predicates/functions as: Name(arg1, arg2)
2. Variables MUST start with a LOWERCASE letter (e.g., x, y, variable1)
3. Constants/Predicates MUST start with an UPPERCASE letter (e.g., John, Apple, Riya)
Enter Expression A (e.g., Eats(x, Apple)): p(g(z),f(g(y)),z)
Enter Expression B (e.g., Eats(Riya, y)): p(g(z),f(w),h(john))
------------------------------------------------------------
Result: UNIFICATION SUCCESS ✅

Most General Unifier (MGU) Substitutions:
  w -> g(y)
  z -> h(john)

Final Unified Expression:
Parsing Error: Please check your syntax/parentheses format. (unhashable type: 'list')
